# 01 Data overview


## Problem statement

The dataset holds 101,766 inpatient encounters of diabetic patients from 130 US hospitals (1999-2008).
Question: which patient, admission and treatment characteristics go with hospital readmission?

- **Target:** `readmitted` has 3 classes: `NO`, `>30`, `<30`.
- **Main outcome for the analysis:** `readmitted_30` = `readmitted == "<30"`. Readmission within 30 days is the clinically relevant outcome.
- **Unit of analysis:** one encounter. A patient can have several encounters, checked below.


In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import seaborn as sns

from eda_diabetes.data import (
    EXPIRED_HOSPICE_DISCHARGE,
    ID_MAPPINGS,
    MISSING_ID_CODES,
    NEAR_CONSTANT_COLUMNS,
    load_raw,
)
from eda_diabetes.paths import DATA_RAW

sns.set_theme(style="whitegrid")

## Data dictionary

Source: `data/raw/variables.csv`, grouped by kind.


In [ ]:
variables = pd.read_csv(DATA_RAW / "variables.csv")

demographics = ["race", "gender", "age", "weight"]
admission = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "payer_code",
    "medical_specialty",
    "time_in_hospital",
]
utilization = [
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
]
diagnoses = ["diag_1", "diag_2", "diag_3", "number_diagnoses"]
labs = ["max_glu_serum", "A1Cresult"]
other = ["change", "diabetesMed"]


def kind(name: str) -> str:
    groups = {
        "demographics": demographics,
        "admission": admission,
        "utilization": utilization,
        "diagnoses": diagnoses,
        "labs": labs,
        "other": other,
    }
    for group, cols in groups.items():
        if name in cols:
            return group
    if variables.set_index("name").loc[name, "role"] == "ID":
        return "id"
    if name == "readmitted":
        return "target"
    return "medication"


variables["group"] = variables["name"].map(kind)
variables.groupby("group")["name"].agg(["count", list])

In [ ]:
pd.set_option("display.max_colwidth", 120)
variables.loc[variables["group"] != "medication", ["group", "name", "type", "description"]]

The 23 medication columns share one description: `No`, `Steady`, `Up`, `Down` for the drug during the encounter.


## Shape, dtypes, info


In [ ]:
df = load_raw()
df.shape

In [ ]:
df.info()

## Identifiers and repeat patients

`encounter_id` is unique per row. `patient_nbr` is not: the same patient appears in several encounters.
Rows of one patient are not independent, which matters for any readmission analysis.


In [ ]:
print(f"rows: {len(df):,}")
print(f"unique encounter_id: {df['encounter_id'].nunique():,}")
print(f"unique patient_nbr: {df['patient_nbr'].nunique():,}")

In [ ]:
encounters_per_patient = df.groupby("patient_nbr").size()
encounters_per_patient.value_counts().sort_index().head(10).to_frame("n_patients")

## Summary statistics


In [ ]:
df.drop(columns=["encounter_id", "patient_nbr"]).describe().T

In [ ]:
df.describe(include="object").T

## Cardinality and constant columns

Columns with 1 value carry no information. Columns with very many levels need grouping.


In [ ]:
cardinality = df.nunique().sort_values().to_frame("n_unique")
top_share = {c: df[c].value_counts(normalize=True).iloc[0].round(4) for c in cardinality.index}
cardinality["top_share"] = pd.Series(top_share)
cardinality.query("n_unique <= 2 or n_unique > 20")

In [ ]:
# 'No' share per medication column: near-constant ones are dropped in cleaning.
print(NEAR_CONSTANT_COLUMNS)
df[NEAR_CONSTANT_COLUMNS].apply(lambda s: s.value_counts().to_dict())

## Missing placeholder

`scripts/download_data.py` uses `ucimlrepo`, which already returns missing values as NaN. The original CSV used `?`.
Check that no `?` is left.


In [ ]:
(df == "?").sum().sum()

## Target: `readmitted`


In [ ]:
df["readmitted"].value_counts(dropna=False)

In [ ]:
sns.countplot(data=df, x="readmitted", order=["NO", ">30", "<30"]);

In [ ]:
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)
df["readmitted_30"].mean().round(3)

Class `<30` is the minority, so the binary target is imbalanced.


## Missing values


In [ ]:
missing = df.isna().sum().loc[lambda s: s > 0].to_frame("n_missing")
missing["pct"] = (missing["n_missing"] / len(df) * 100).round(1)
missing.sort_values("n_missing", ascending=False)

What NaN means differs per column:

- `max_glu_serum`, `A1Cresult`: test not performed. Not the same as unknown. Becomes category `None` in cleaning.
- `weight` (97% missing): too sparse to use. Dropped in cleaning.
- `payer_code`, `medical_specialty`, `race`: unknown. Becomes `Unknown` in cleaning.
- `diag_1`, `diag_2`, `diag_3`: few rows. Becomes `Unknown` in the ICD-9 grouping.


## Missing values hidden in codes

Some ID codes mean missing (`NULL`, `Not Available`, `Not Mapped`, `Unknown/Invalid`) but are not NaN.


In [ ]:
hidden = {col: int(df[col].isin(codes).sum()) for col, codes in MISSING_ID_CODES.items()}
hidden["gender == Unknown/Invalid"] = int((df["gender"] == "Unknown/Invalid").sum())
pd.Series(hidden, name="n_rows")

## Rows to exclude

Discharge to hospice or death means the patient cannot be readmitted. These rows bias the target.


In [ ]:
expired_hospice = df["discharge_disposition_id"].isin(EXPIRED_HOSPICE_DISCHARGE)
print(f"expired or hospice: {expired_hospice.sum():,}")
df.loc[expired_hospice, "readmitted"].value_counts()

## ID code mappings


In [ ]:
for col, mapping in ID_MAPPINGS.items():
    display(df[col].map(mapping).value_counts(dropna=False).head(10).to_frame(col))